In [5]:
import pandas as pd
import os
from pathlib import Path

# 1. Configuración de rutas
# Rutas relativas subiendo un nivel (..)
input_folder = "../data/raw"
filename = "participacion_total_hs6_countries.xlsx"

# Construcción de la ruta de entrada
input_path = Path(input_folder) / filename

# Configuración de salida
output_folder = "../data/intermediate"
output_filename = "Participacion_HTS_Completo.xlsx"
output_path = Path(output_folder) / output_filename

# Crear la carpeta de salida si no existe para evitar errores
os.makedirs(output_folder, exist_ok=True)

# 2. Cargar el archivo
print(f"Leyendo archivo desde: {input_path}")
try:
    df = pd.read_excel(input_path)
except ValueError:
    df = pd.read_csv(input_path)
except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {input_path}")
    print("Verifica que estés ejecutando el script desde la carpeta correcta.")
    exit()

# 3. Asegurarse de que la columna Fecha sea de tipo datetime
df['Fecha'] = pd.to_datetime(df['Fecha'])

# 4. Definir el rango completo de fechas
min_date = df['Fecha'].min()
max_date = df['Fecha'].max()
print(f"Rango de fechas: {min_date.date()} a {max_date.date()}")

all_dates = pd.date_range(start=min_date, end=max_date, freq='MS')

# 5. Crear la estructura base (scaffold)
unique_subpartidas = df['Subpartida'].unique()

multi_index = pd.MultiIndex.from_product(
    [unique_subpartidas, all_dates], 
    names=['Subpartida', 'Fecha']
)

df_base = pd.DataFrame(index=multi_index).reset_index()

# 6. Unir con los datos originales
df_merged = pd.merge(df_base, df, on=['Subpartida', 'Fecha'], how='left')

# 7. Calcular porcentajes
# Las filas vacías seguirán siendo NaN en estas columnas nuevas
df_merged['Mexico (%)'] = df_merged['Mexico'] / df_merged['Total']
df_merged['China (%)'] = df_merged['China'] / df_merged['Total']

# 8. Ordenar y guardar
df_merged = df_merged.sort_values(by=['Subpartida', 'Fecha'])

print(f"Guardando archivo en: {output_path}")
df_merged.to_excel(output_path, index=False)
print("¡Proceso terminado exitosamente!")

Leyendo archivo desde: ..\data\raw\participacion_total_hs6_countries.xlsx
Rango de fechas: 2018-01-01 a 2025-10-01
Guardando archivo en: ..\data\intermediate\Participacion_HTS_Completo.xlsx
¡Proceso terminado exitosamente!
